## Step 1: Setup and Configuration

In [1]:
!uv pip install unsloth

Using Python 3.10.19 environment at: /opt/miniconda3/envs/khoina_llamafactory
Audited 1 package in 26ms


In [2]:
%cd /home/khoina/LLaMA-OSS

/home/khoina/LLaMA-OSS


In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
from pathlib import Path

# Set paths
LLAMA_FACTORY_DIR = Path("LLaMA-Factory")
os.chdir(LLAMA_FACTORY_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"LLaMA-Factory directory: {LLAMA_FACTORY_DIR}")
try:
    import unsloth
    from unsloth import FastLanguageModel, is_bfloat16_supported
    print(f"✓ Unsloth imported successfully")
    print(f"  Version: {unsloth.__version__}")
except ImportError as e:
    print("✗ Unsloth not found!")
    print("  Install with: pip install \"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git\"")
    raise e
try:
    from trl import GRPOConfig, GRPOTrainer
    print("✓ TRL imported successfully (GRPO support)")
except ImportError as e:
    print("✗ TRL not found!")
    print("  Install with: pip install trl")
    raise e

print("\nAll dependencies ready for Unsloth GRPO training!")

Working directory: /home/khoina/LLaMA-OSS/LLaMA-Factory
LLaMA-Factory directory: LLaMA-Factory
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/khoina/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-12 04:24:38 [__init__.py:216] Automatically detected platform cuda.


W1112 04:24:38.642000 1252570 site-packages/torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1112 04:24:38.642000 1252570 site-packages/torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
2025-11-12 04:24:38,647 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth imported successfully
  Version: 2025.11.2
✓ TRL imported successfully (GRPO support)

All dependencies ready for Unsloth GRPO training!


## Step 2: Configure Model and Training Parameters

In [ ]:
# Model configuration
MODEL_NAME_OR_PATH = "output/llama3_2_3b_sft_concat_sorted"
MODEL_MAX_LENGTH = 5120

# Training configuration
OUTPUT_DIR = "saves/llama3-3b/grpo_unsloth_3modes"

# Dataset
DATA_DIR = Path("data")
GRPO_LOW_PATH = DATA_DIR / "combined_grpo_train_low.jsonl"
GRPO_MEDIUM_PATH = DATA_DIR / "combined_grpo_train_medium.jsonl"
GRPO_HIGH_PATH = DATA_DIR / "combined_grpo_train_high.jsonl"
VAL_PATH = DATA_DIR / "combined_val.jsonl"

# Performance settings
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
# NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 5.0e-6
MAX_STEPS = 10

# LoRA settings
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# GRPO
NUM_GENERATIONS = 4
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 4096
MAX_PROMPT_LENGTH = MODEL_MAX_LENGTH - MAX_NEW_TOKENS
REASONING_TARGETS = {
    "low": {"min_tokens": 0, "max_tokens": 512, "weight": 1.0},
    "medium": {"min_tokens": 0, "max_tokens": 2048, "weight": 1.0},
    "high": {"min_tokens": 0, "max_tokens": 4096, "weight": 1.0}
}

print("="*70)
print("Configuration Summary")
print("="*70)
print(f"Model: {Path(MODEL_NAME_OR_PATH).name}")
print(f"Output: {OUTPUT_DIR}")
print(f"\nDatasets (3 modes):")
print(f"  • Low:    {GRPO_LOW_PATH.name}")
print(f"  • Medium: {GRPO_MEDIUM_PATH.name}")
print(f"  • High:   {GRPO_HIGH_PATH.name}")
print(f"\nTraining:")
print(f"  • Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  • Gradient accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Effective batch: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Generations/prompt: {NUM_GENERATIONS}")
print("="*70)

Configuration Summary
Model: llama3_2_3b_sft_concat_sorted
Output: saves/llama3-3b/grpo_unsloth_3modes

Datasets (3 modes):
  • Low:    combined_grpo_train_low.jsonl
  • Medium: combined_grpo_train_medium.jsonl
  • High:   combined_grpo_train_high.jsonl

Training:
  • Batch size: 4
  • Gradient accum: 4
  • Effective batch: 16
  • Learning rate: 5e-06
  • Generations/prompt: 4


## Step 3: Load and Combine 3 GRPO Datasets

In [5]:
import json
from datasets import Dataset, concatenate_datasets

def load_jsonl_with_mode(filepath, mode_name):
    """Load JSONL and add mode information."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            if 'mode' not in item:
                item['mode'] = mode_name
            data.append(item)
    return data

print("Loading 3 GRPO datasets...")
print("="*60)

low_data = load_jsonl_with_mode(GRPO_LOW_PATH, "low")
print(f"✓ Low mode:    {len(low_data):,} examples")

medium_data = load_jsonl_with_mode(GRPO_MEDIUM_PATH, "medium")
print(f"✓ Medium mode: {len(medium_data):,} examples")

high_data = load_jsonl_with_mode(GRPO_HIGH_PATH, "high")
print(f"✓ High mode:   {len(high_data):,} examples")

combined_data = low_data + medium_data + high_data
print(f"\n✓ Combined:    {len(combined_data):,} total examples")

train_dataset = Dataset.from_list(combined_data)
print(f"\n✓ Created HuggingFace Dataset with {len(train_dataset)} examples")
print(f"  Columns: {train_dataset.column_names}")

val_data = load_jsonl_with_mode(VAL_PATH, "mixed")
eval_dataset = Dataset.from_list(val_data)
print(f"\n✓ Validation:  {len(eval_dataset):,} examples")

print("\n" + "="*60)
print("Dataset Distribution:")
print("="*60)
mode_counts = {"low": len(low_data), "medium": len(medium_data), "high": len(high_data)}
for mode, count in mode_counts.items():
    pct = 100 * count / len(combined_data)
    print(f"  {mode:8s}: {count:6,} ({pct:5.1f}%)")
print("="*60)

Loading 3 GRPO datasets...
✓ Low mode:    8,897 examples
✓ Medium mode: 9,450 examples
✓ High mode:   8,790 examples

✓ Combined:    27,137 total examples

✓ Created HuggingFace Dataset with 27137 examples
  Columns: ['prompt', 'response', 'mode', 'reasoning_tokens', 'dataset', 'reasoning', 'combined_index']

✓ Validation:  1,574 examples

Dataset Distribution:
  low     :  8,897 ( 32.8%)
  medium  :  9,450 ( 34.8%)
  high    :  8,790 ( 32.4%)


## Step 4: Load Model with Unsloth + Add LoRA

In [6]:
from unsloth import FastLanguageModel
import torch

print("Loading model with Unsloth...")
print("="*60)

# Load SFT model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME_OR_PATH,
    max_seq_length=MODEL_MAX_LENGTH,
    dtype=None,
    load_in_4bit=False,
    fast_inference=True,
)

print(f"✓ Loaded model: {Path(MODEL_NAME_OR_PATH).name}")
print(f"✓ Max sequence length: {MODEL_MAX_LENGTH}")
print(f"✓ Device: {next(model.parameters()).device}")
print(f"✓ Dtype: {next(model.parameters()).dtype}")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print(f"\n✓ Added LoRA adapters:")
print(f"  • Rank: {LORA_RANK}")
print(f"  • Alpha: {LORA_ALPHA}")
print(f"  • Dropout: {LORA_DROPOUT}")
print(f"  • Gradient checkpointing: unsloth (optimized)")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
print("="*60)

Loading model with Unsloth...
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.0.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]


output/llama3_2_3b_sft_concat_sorted does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
✓ Loaded model: llama3_2_3b_sft_concat_sorted
✓ Max sequence length: 5120
✓ Device: cuda:0
✓ Dtype: torch.bfloat16


Unsloth 2025.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✓ Added LoRA adapters:
  • Rank: 16
  • Alpha: 16
  • Dropout: 0.0
  • Gradient checkpointing: unsloth (optimized)

Trainable parameters: 24,313,856 / 3,237,063,680 (0.75%)


## Step 5: Create Multi-Mode Reward Function for 3 Reasoning Modes

In [8]:
import re
import torch
import math
from typing import Dict

class BalancedMultiModeRewardFunction:
    """Reward function with Q1/Q3-based outlier removal and auto min/max"""

    def __init__(self, mode_statistics, 
                 correctness_weight=0.6, length_weight=0.4,
                 iqr_multiplier=1.5,
                 target_strategy="mean_std"):
        """
        Args:
            mode_statistics: Dict of mode stats with Q1, Q3, mean, std, etc.
            correctness_weight: Weight for correctness reward (0-1)
            length_weight: Weight for length reward (0-1)
            iqr_multiplier: Multiplier for IQR (1.5 = standard, 3.0 = extreme outliers)
            target_strategy: Strategy to compute min/max targets
                - "mean_std": min = mean - std, max = mean + std
                - "q1_q3": min = Q1, max = Q3
                - "percentile": min = 10th percentile, max = 90th percentile
                - "actual": min = actual min, max = actual max from data
        """
        self.mode_statistics = mode_statistics
        self.correctness_weight = correctness_weight
        self.length_weight = length_weight
        self.iqr_multiplier = iqr_multiplier
        self.target_strategy = target_strategy
        self.__name__ = "BalancedMultiModeRewardFunction"
        
        # Compute mode targets from statistics
        self._compute_mode_targets()
        
        # Compute outlier bounds
        self._compute_outlier_bounds()

    def _compute_mode_targets(self):
        """Compute min/max targets for each mode based on statistics"""
        self.mode_targets = {}
        
        print("\nMode Targets (auto-computed):")
        print("-" * 60)
        
        for mode in self.mode_statistics:
            stats = self.mode_statistics[mode]
            
            if self.target_strategy == "mean_std":
                # Use mean ± 1 std
                min_tok = max(0, stats['mean'] - stats['std'])
                max_tok = stats['mean'] + stats['std']
            
            elif self.target_strategy == "q1_q3":
                # Use Q1 and Q3
                min_tok = stats['p25']
                max_tok = stats['p75']
            
            elif self.target_strategy == "percentile":
                # Use 10th and 90th percentile (if available)
                min_tok = stats.get('p10', stats['p25'])
                max_tok = stats.get('p90', stats['p75'])
            
            elif self.target_strategy == "actual":
                # Use actual min/max from data
                min_tok = stats['min']
                max_tok = stats['max']
            
            else:
                raise ValueError(f"Unknown target_strategy: {self.target_strategy}")
            
            self.mode_targets[mode] = {
                'min_tokens': int(min_tok),
                'max_tokens': int(max_tok)
            }
            
            print(f"{mode.upper():8s}: [{self.mode_targets[mode]['min_tokens']}, "
                  f"{self.mode_targets[mode]['max_tokens']}]  "
                  f"(strategy: {self.target_strategy})")

    def _compute_outlier_bounds(self):
        """Compute outlier bounds using IQR method for each mode"""
        self.outlier_bounds = {}
        
        print("\nOutlier Bounds (IQR Method):")
        print("-" * 60)
        
        for mode in self.mode_statistics:
            q1 = self.mode_statistics[mode]['p25']
            q3 = self.mode_statistics[mode]['p75']
            iqr = q3 - q1
            
            lower_bound = q1 - self.iqr_multiplier * iqr
            upper_bound = q3 + self.iqr_multiplier * iqr
            
            # Ensure bounds are positive
            lower_bound = max(0, lower_bound)
            
            self.outlier_bounds[mode] = {
                'lower': lower_bound,
                'upper': upper_bound,
                'q1': q1,
                'q3': q3,
                'iqr': iqr
            }
            
            print(f"{mode.upper():8s}: [{lower_bound:.0f}, {upper_bound:.0f}]  "
                  f"(Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f})")

    def count_reasoning_tokens(self, text):
        """Count tokens in <think> tags."""
        matches = re.findall(r"<think>(.*?)</think>", text, re.DOTALL)
        if not matches:
            return 0
        reasoning_text = " ".join(matches)
        tokens = tokenizer(reasoning_text, add_special_tokens=False)
        return len(tokens['input_ids']) + 5  # +5 for <think><\think> tokens

    def extract_answer(self, text):
        """Extract answer from \\boxed{}."""
        matches = re.findall(r"\\boxed\{([^}]+)\}", text)
        return matches[-1].strip() if matches else ""

    def normalize_answer(self, answer):
        """Normalize answer for comparison."""
        answer = answer.lower().strip()
        answer = answer.replace(" ", "").replace(",", "")
        return answer

    def check_correctness(self, prediction, ground_truth):
        """Check if prediction matches ground truth."""
        if not ground_truth or not prediction:
            return 0.0
            
        pred_norm = self.normalize_answer(prediction)
        truth_norm = self.normalize_answer(ground_truth)
        
        if pred_norm == truth_norm:
            return 1.0
        
        return 0.0

    def compute_length_reward_gaussian(self, tokens, target_mode):
        """
        Gaussian-based length reward with Q1/Q3-based outlier removal.
        
        Outliers are removed using IQR method:
        - Lower bound: Q1 - k*IQR
        - Upper bound: Q3 + k*IQR
        """
        config = self.mode_targets[target_mode]
        bounds = self.outlier_bounds[target_mode]
        
        # Check if outlier
        if tokens < bounds['lower'] or tokens > bounds['upper']:
            return 0.0
        
        # Use target min/max for Gaussian center
        min_tok = config["min_tokens"]
        max_tok = config["max_tokens"]
        
        # Center and range
        center = (min_tok + max_tok) / 2
        range_size = max_tok - min_tok
        
        # Gaussian with sigma = range/4 (95% within [min, max])
        sigma = range_size / 4
        
        # Gaussian reward
        distance = abs(tokens - center)
        reward = math.exp(-(distance ** 2) / (2 * sigma ** 2))
        
        return reward

    def detect_mode_from_prompt(self, prompt):
        """Extract mode from prompt with format: <mode: low|medium|high>"""
        match = re.search(r"<mode:\s*(low|medium|high)>", prompt)
        return match.group(1) if match else "medium"

    def __call__(self, prompts, completions, modes=None, ground_truths=None, **kwargs):
        """
        Compute balanced rewards with Q1/Q3-based outlier removal.

        Args:
            prompts: List of input prompts
            completions: List of model outputs
            modes: List of target modes (optional, will auto-detect if None)
            ground_truths: List of correct answers (optional)
        Returns:
            Tensor of reward scores
        """
        rewards = []

        for i, completion in enumerate(completions):
            # Get mode
            if modes is not None and i < len(modes):
                target_mode = modes[i]
            else:
                target_mode = self.detect_mode_from_prompt(prompts[i])
            
            # Component 1: Length reward
            tokens = self.count_reasoning_tokens(completion)
            length_reward = self.compute_length_reward_gaussian(tokens, target_mode)
            
            # Component 2: Correctness reward
            correctness_reward = 0.0
            if ground_truths is not None and i < len(ground_truths):
                predicted_answer = self.extract_answer(completion)
                correctness_reward = self.check_correctness(predicted_answer, ground_truths[i])
            
            # Combined reward
            if ground_truths is not None:
                total_reward = (
                    self.correctness_weight * correctness_reward +
                    self.length_weight * length_reward
                )
            else:
                total_reward = length_reward

            rewards.append(total_reward)

        return torch.tensor(rewards, dtype=torch.float32)

In [ ]:
from dataclasses import dataclass
from collections import defaultdict, Counter
from typing import Dict, List, Optional
import numpy as np
import re

@dataclass
class RewardConfig:
    """Configuration for adaptive reward function"""
    # Reward 1: Format + Correctness weights
    w_correctness: float = 1.0
    w_format: float = 0.3
    
    # Reward 2: Adaptive Budget
    w_budget: float = 0.5
    
    # Reward 3: Repeat Penalty
    w_repeat_penalty: float = 0.3
    ngram_sizes: tuple = (3, 4, 5)
    repetition_threshold: float = 0.3
    
    # Mode statistics (pre-computed from data)
    mode_stats: Dict[str, Dict[str, float]] = None
    
    # Reward clipping
    max_reward: float = 2.0
    min_reward: float = -1.0


class AdaptiveRewardFunction:
    """
    3 Reward Functions for GRPO:
    1. Format + Correctness: <think> tags + \\box{} + correct answer
    2. Adaptive Budget: Penalize deviation from mode statistics
    3. Repeat Penalty: Penalize repetitive reasoning
    """
    
    def __init__(self, config: RewardConfig, tokenizer=None):
        self.config = config
        self.tokenizer = tokenizer
    
    # ========================================================================
    # REWARD 1: Format + Correctness
    # ========================================================================
    
    def extract_thinking(self, text: str):
        """Extract thinking content from <think>...</think>"""
        match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
        if match:
            return True, match.group(1).strip()
        return False, ""
    
    def extract_answer(self, completion: str):
        """Extract answer from \\box{} after </think>"""
        has_think = '</think>' in completion
        if not has_think:
            return None
        
        # Find \\box{} after </think>
        think_end = completion.find('</think>')
        after_think = completion[think_end:]
        
        match = re.search(r'\\box\{([^}]+)\}', after_think)
        if match:
            return self._normalize_answer(match.group(1))
        return None
    
    def _normalize_answer(self, answer: str):
        """Normalize answer for comparison"""
        answer = answer.strip()
        # Remove formatting
        answer = answer.replace("\\%", "").replace("^\\circ", "").replace("\\$", "")
        answer = answer.replace("\\pi", "").replace(" ", "")
        answer = answer.replace("%", "").replace("^circ", "").replace("$", "").replace("pi", "")
        
        # Handle fractions
        if '/' in answer and '\\' not in answer:
            try:
                parts = answer.split('/')
                answer = str(float(parts[0]) / float(parts[1]))
            except:
                pass
        return answer.lower()
    
    def verify_correctness(self, completion: str, ground_truth: str):
        """Check if answer matches ground truth"""
        extracted = self.extract_answer(completion)
        if extracted is None:
            return False
        
        truth_normalized = self._normalize_answer(ground_truth)
        
        # Exact match
        if extracted == truth_normalized:
            return True
        
        # Numerical tolerance
        try:
            val_extract = float(extracted)
            val_truth = float(truth_normalized)
            if abs(val_extract - val_truth) / max(abs(val_truth), 1e-10) < 0.001:
                return True
        except:
            pass
        
        return False
    
    def compute_format_correctness_reward(self, completion: str, ground_truth: str):
        """
        Reward 1: Format + Correctness
        Returns: (reward, is_correct)
        """
        reward = 0.0
        
        # Check format
        has_thinking, _ = self.extract_thinking(completion)
        has_box = self.extract_answer(completion) is not None
        
        if has_thinking and has_box:
            reward += self.config.w_format
        
        # Check correctness
        is_correct = self.verify_correctness(completion, ground_truth)
        if is_correct:
            reward += self.config.w_correctness
        
        return reward, is_correct
    
    # ========================================================================
    # REWARD 2: Adaptive Budget
    # ========================================================================
    
    def count_tokens(self, text: str) -> int:
        """Count tokens using tokenizer or approximate"""
        if self.tokenizer is not None:
            return len(self.tokenizer(text, add_special_tokens=False)['input_ids'])
        else:
            # Approximate: words * 1.3 for BPE
            return int(len(text.split()) * 1.3)
    
    def count_thinking_tokens(self, completion: str) -> int:
        """Count tokens inside <think>...</think>"""
        has_thinking, thinking_text = self.extract_thinking(completion)
        if has_thinking:
            return self.count_tokens(thinking_text)
        return 0
    
    def compute_budget_reward(self, completion: str, mode: str, is_correct: bool):
        """
        Reward 2: Adaptive Budget
        Rewards being near mode mean, penalizes exceeding mean
        Allows shorter if correct
        """
        if mode not in self.config.mode_stats:
            return 0.0
        
        stats = self.config.mode_stats[mode]
        mean = stats['mean']
        std = stats['std']
        
        token_count = self.count_thinking_tokens(completion)
        
        if token_count == 0:
            return -self.config.w_budget  # No thinking
        
        deviation = token_count - mean
        
        # Within 1 std: high reward
        if abs(deviation) <= std:
            reward = self.config.w_budget * (1.0 - abs(deviation) / std * 0.3)
        
        # Shorter than mean
        elif deviation < 0:
            if is_correct:
                reward = self.config.w_budget * 0.9  # Good!
            else:
                reward = self.config.w_budget * (0.5 - abs(deviation) / mean * 0.5)
        
        # Longer than mean: penalty
        else:
            excess = (token_count - mean) / std
            penalty = 0.1 * (excess ** 1.5)
            reward = -penalty
        
        return reward
    
    # ========================================================================
    # REWARD 3: Repeat Penalty
    # ========================================================================
    
    def compute_ngram_repetition(self, text: str, n: int) -> float:
        """Compute n-gram repetition ratio"""
        tokens = text.split()
        if len(tokens) < n:
            return 0.0
        
        ngrams = [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        if len(ngrams) == 0:
            return 0.0
        
        ngram_counts = Counter(ngrams)
        repeated = sum(count - 1 for count in ngram_counts.values() if count > 1)
        
        return repeated / len(ngrams)
    
    def compute_token_repetition(self, text: str) -> float:
        """Compute token-level repetition"""
        tokens = text.split()
        if len(tokens) == 0:
            return 0.0
        
        token_counts = Counter(tokens)
        repeated = sum(count - 1 for count in token_counts.values() if count > 1)
        
        return repeated / len(tokens)
    
    def compute_repeat_penalty(self, completion: str):
        """
        Reward 3: Repetition Penalty
        Penalizes repeated words/phrases in thinking
        """
        has_thinking, thinking_text = self.extract_thinking(completion)
        if not has_thinking:
            return 0.0
        
        # Check repetition at different levels
        max_repetition = 0.0
        
        # Token-level
        token_rep = self.compute_token_repetition(thinking_text)
        max_repetition = max(max_repetition, token_rep)
        
        # N-gram level
        for n in self.config.ngram_sizes:
            ngram_rep = self.compute_ngram_repetition(thinking_text, n)
            max_repetition = max(max_repetition, ngram_rep)
        
        # Apply penalty if exceeds threshold
        if max_repetition > self.config.repetition_threshold:
            penalty_strength = (max_repetition - self.config.repetition_threshold) / \
                              (1.0 - self.config.repetition_threshold)
            penalty = -self.config.w_repeat_penalty * (penalty_strength ** 2)
            return penalty
        
        return 0.0
    
    # ========================================================================
    # Combined Reward
    # ========================================================================
    
    def compute_reward(
        self,
        prompt: str,
        completion: str,
        ground_truth: str,
        problem_id: str,
        group_completions: List[str],
        mode: Optional[str] = None
    ) -> float:
        """
        Main reward computation combining all 3 rewards
        
        Args:
            prompt: Problem statement (unused but kept for compatibility)
            completion: Model's solution
            ground_truth: Correct answer
            problem_id: Unique identifier (unused but kept for compatibility)
            group_completions: All completions (unused but kept for compatibility)
            mode: Reasoning mode ('low', 'medium', 'high')
        
        Returns:
            reward: Total reward value
        """
        if mode is None:
            mode = 'medium'  # Default
        
        # Reward 1: Format + Correctness
        r1, is_correct = self.compute_format_correctness_reward(completion, ground_truth)
        
        # Reward 2: Budget
        r2 = self.compute_budget_reward(completion, mode, is_correct)
        
        # Reward 3: Repetition
        r3 = self.compute_repeat_penalty(completion)
        
        # Total
        total_reward = r1 + r2 + r3
        
        # Clip
        total_reward = np.clip(
            total_reward,
            self.config.min_reward,
            self.config.max_reward
        )
        
        return total_reward
    
    def compute_group_rewards(
        self,
        prompts: List[str],
        completions: List[str],
        ground_truths: List[str],
        problem_ids: List[str],
        mode: Optional[str] = None
    ) -> np.ndarray:
        """
        Compute rewards for a group of completions (for GRPO)
        
        Args:
            prompts: List of problem statements
            completions: List of solutions
            ground_truths: List of correct answers
            problem_ids: List of problem identifiers
            mode: Reasoning mode
        
        Returns:
            rewards: Array of reward values
        """
        batch_size = len(completions)
        rewards = np.zeros(batch_size)
        
        # Group by problem (for compatibility)
        problem_groups = defaultdict(list)
        for i, pid in enumerate(problem_ids):
            problem_groups[pid].append(completions[i])
        
        # Compute rewards
        for i in range(batch_size):
            group_comps = problem_groups[problem_ids[i]]
            rewards[i] = self.compute_reward(
                prompt=prompts[i],
                completion=completions[i],
                ground_truth=ground_truths[i],
                problem_id=problem_ids[i],
                group_completions=group_comps,
                mode=mode
            )
        
        return rewards
    
    def get_statistics(self) -> Dict[str, float]:
        """Get training statistics (simplified)"""
        stats = {
            'mode_stats': self.config.mode_stats,
            'w_correctness': self.config.w_correctness,
            'w_format': self.config.w_format,
            'w_budget': self.config.w_budget,
            'w_repeat_penalty': self.config.w_repeat_penalty,
        }
        return stats
    
    def compute_reward_breakdown(
        self,
        completion: str,
        ground_truth: str,
        mode: str
    ) -> Dict[str, float]:
        """
        Get detailed reward breakdown for debugging
        """
        r1, is_correct = self.compute_format_correctness_reward(completion, ground_truth)
        r2 = self.compute_budget_reward(completion, mode, is_correct)
        r3 = self.compute_repeat_penalty(completion)
        
        return {
            'format_correctness': r1,
            'budget': r2,
            'repetition': r3,
            'total': np.clip(r1 + r2 + r3, self.config.min_reward, self.config.max_reward),
            'is_correct': is_correct,
            'thinking_tokens': self.count_thinking_tokens(completion),
        }


# ============================================================================
# Helper: Compute mode statistics from data
# ============================================================================

def compute_mode_statistics_from_data(
    data: List[Dict],
    tokenizer=None
) -> Dict[str, Dict[str, float]]:
    """
    Compute statistics for each mode from your dataset
    
    Args:
        data: List of dicts with 'completion' and 'mode' keys
        tokenizer: Optional tokenizer for accurate token counting
    
    Returns:
        Dict with statistics for each mode
    """
    mode_tokens = {'low': [], 'medium': [], 'high': []}
    
    # Temporary reward function for counting
    temp_config = RewardConfig()
    temp_reward = AdaptiveRewardFunction(temp_config, tokenizer)
    
    for item in data:
        mode = item.get('mode', 'medium')
        completion = item.get('completion', '')
        
        token_count = temp_reward.count_thinking_tokens(completion)
        if token_count > 0:
            mode_tokens[mode].append(token_count)
    
    stats = {}
    for mode in ['low', 'medium', 'high']:
        tokens = mode_tokens[mode]
        if tokens:
            stats[mode] = {
                'mean': float(np.mean(tokens)),
                'std': float(np.std(tokens)),
                'min': float(np.min(tokens)),
                'max': float(np.max(tokens)),
                'median': float(np.median(tokens)),
                'count': len(tokens),
            }
        else:
            # Defaults
            defaults = {'low': 150, 'medium': 350, 'high': 1000}
            stats[mode] = {
                'mean': float(defaults[mode]),
                'std': 50.0 if mode == 'low' else (100.0 if mode == 'medium' else 500.0),
                'min': 50.0,
                'max': 500.0 if mode == 'low' else (700.0 if mode == 'medium' else 4000.0),
                'median': float(defaults[mode]),
                'count': 0,
            }
    
    return stats


# ============================================================================
# Usage Example
# ============================================================================

if __name__ == "__main__":
    # Example: Load mode stats from your data
    example_data = [
        {
            'completion': '<think>2+2=4</think>\\box{4}',
            'mode': 'low'
        },
        # Add your data here
    ]
    
    mode_stats = compute_mode_statistics_from_data(example_data)
    
    # Initialize reward function
    config = RewardConfig(
        w_correctness=1.0,
        w_format=0.3,
        w_budget=0.5,
        w_repeat_penalty=0.3,
        mode_stats=mode_stats,
    )
    
    reward_func = AdaptiveRewardFunction(config)
    
    # Test
    test_completion = """<think>
First, add 2 and 3.
2 + 3 = 5
Therefore, the answer is 5.
</think>
\\box{5}"""
    
    breakdown = reward_func.compute_reward_breakdown(
        completion=test_completion,
        ground_truth="5",
        mode="medium"
    )
    
    print("Reward Breakdown:")
    for key, value in breakdown.items():
        print(f"  {key}: {value}")

In [9]:
# p = "<think></think>"
# tokens = tokenizer(p, add_special_tokens=False)
# len(tokens['input_ids'])

In [10]:
from pathlib import Path
import json
import numpy as np

# Step 1: Calculate statistics from your data
data_dir = Path("data")
grpo_files = [
    "combined_grpo_train_low.jsonl",
    "combined_grpo_train_medium.jsonl",
    "combined_grpo_train_high.jsonl",
]

mode_statistics = {}

print("Computing mode statistics from data...")
print("="*80)

for filename in grpo_files:
    filepath = data_dir / filename
    if filepath.exists():
        reasoning_tokens_list = []
        
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                if 'reasoning_tokens' in data:
                    reasoning_tokens = tokenizer(data['response'], add_special_tokens=False)
                    reasoning_tokens_list.append(len(reasoning_tokens['input_ids']))
        
        if reasoning_tokens_list:
            tokens_array = np.array(reasoning_tokens_list)
            mode = filename.replace('combined_grpo_train_', '').replace('.jsonl', '')
            
            mode_statistics[mode] = {
                'count': len(tokens_array),
                'min': int(tokens_array.min()),
                'max': int(tokens_array.max()),
                'mean': tokens_array.mean(),
                'std': tokens_array.std(),
                'median': np.median(tokens_array),
                'p10': np.percentile(tokens_array, 10),
                'p25': np.percentile(tokens_array, 25),  # Q1
                'p75': np.percentile(tokens_array, 75),  # Q3
                'p90': np.percentile(tokens_array, 90),
            }
            
            print(f"\n{mode.upper()} Mode:")
            print(f"  Count: {mode_statistics[mode]['count']:,}")
            print(f"  Range: [{mode_statistics[mode]['min']}, {mode_statistics[mode]['max']}]")
            print(f"  Mean ± Std: {mode_statistics[mode]['mean']:.1f} ± {mode_statistics[mode]['std']:.1f}")
            print(f"  Q1, Median, Q3: [{mode_statistics[mode]['p25']:.0f}, "
                  f"{mode_statistics[mode]['median']:.0f}, {mode_statistics[mode]['p75']:.0f}]")

# Step 2: Initialize reward function with different strategies
print("\n" + "="*80)
print("Initializing Reward Functions with Different Strategies...")
print("="*80)

# Strategy 1: mean ± std (Recommended for balanced distribution)
reward_fn_mean_std = BalancedMultiModeRewardFunction(
    mode_statistics=mode_statistics,
    correctness_weight=0.6,
    length_weight=0.4,
    iqr_multiplier=1.5,
    target_strategy="mean_std"
)

print("\n" + "="*80)

# Strategy 2: Q1 to Q3 (Recommended for robust to outliers)
reward_fn_q1_q3 = BalancedMultiModeRewardFunction(
    mode_statistics=mode_statistics,
    correctness_weight=0.6,
    length_weight=0.4,
    iqr_multiplier=1.5,
    target_strategy="q1_q3"
)

print("\n" + "="*80)

# Strategy 3: 10th to 90th percentile
reward_fn_percentile = BalancedMultiModeRewardFunction(
    mode_statistics=mode_statistics,
    correctness_weight=0.6,
    length_weight=0.4,
    iqr_multiplier=1.5,
    target_strategy="percentile"
)

# Choose the one you want to use
reward_fn = reward_fn_mean_std  # or reward_fn_q1_q3, or reward_fn_percentile

print("\n" + "="*80)
print("✓ Reward function ready!")
print(f"  Using strategy: {reward_fn.target_strategy}")
print(f"  Reward composition: {reward_fn.correctness_weight:.0%} correctness + {reward_fn.length_weight:.0%} length")

Computing mode statistics from data...



LOW Mode:
  Count: 8,897
  Range: [19, 542]
  Mean ± Std: 137.1 ± 107.0
  Q1, Median, Q3: [62, 95, 172]

MEDIUM Mode:
  Count: 9,450
  Range: [46, 2211]
  Mean ± Std: 496.3 ± 453.2
  Q1, Median, Q3: [154, 322, 695]

HIGH Mode:
  Count: 8,790
  Range: [53, 4243]
  Mean ± Std: 1136.4 ± 1052.1
  Q1, Median, Q3: [304, 702, 1707]

Initializing Reward Functions with Different Strategies...

Mode Targets (auto-computed):
------------------------------------------------------------
LOW     : [30, 244]  (strategy: mean_std)
MEDIUM  : [43, 949]  (strategy: mean_std)
HIGH    : [84, 2188]  (strategy: mean_std)

Outlier Bounds (IQR Method):
------------------------------------------------------------
LOW     : [0, 337]  (Q1=62, Q3=172, IQR=110)
MEDIUM  : [0, 1506]  (Q1=154, Q3=695, IQR=541)
HIGH    : [0, 3811]  (Q1=304, Q3=1707, IQR=1403)


Mode Targets (auto-computed):
------------------------------------------------------------
LOW     : [62, 172]  (strategy: q1_q3)
MEDIUM  : [154, 695]  (strate

## Step 6: Configure and Run GRPO Training with Unsloth

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    # num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_NEW_TOKENS,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    num_generations=NUM_GENERATIONS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    logging_steps=1,
    save_steps=100,
    max_grad_norm=1.0,
    save_total_limit=3,
    report_to="none",
    seed=42,
    dataloader_num_workers=4,
    shuffle_dataset=True,
    use_vllm=True,
)

print("="*70)
print("GRPO Training Configuration")
print("="*70)
print(f"Output directory: {OUTPUT_DIR}")
# print(f"Epochs: {NUM_TRAIN_EPOCHS}")
print(f"Batch size: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Generations per prompt: {NUM_GENERATIONS}")
print(f"Temperature: {TEMPERATURE}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print("="*70)

# Prepare dataset formatting
def format_prompt(example):
    """Format example for GRPO training."""
    mode = example.get("mode", "medium")
    if mode == "low":
        prompt = f"Fast response required. Minimize reasoning time. Direct answers only.\n{example['prompt']}"
    elif mode == "medium":
        prompt = f"Balanced approach. Think enough but don't overthink. Reasonable depth.\n{example['prompt']}"
    else:  # mode == "high"
        prompt = f"Maximum accuracy focus. Use extensive reasoning. Verify thoroughly. No time limits.\n{example['prompt']}"
    
    return {
        "prompt": prompt,
        "chosen": example.get("response", ""),
        "mode": mode,
    }

# Format datasets
train_dataset_formatted = train_dataset.map(format_prompt, remove_columns=train_dataset.column_names)
eval_dataset_formatted = eval_dataset.map(format_prompt, remove_columns=eval_dataset.column_names)

print(f"\n✓ Formatted {len(train_dataset_formatted)} training examples")
print(f"✓ Formatted {len(eval_dataset_formatted)} validation examples")
print(f"\nDataset columns: {train_dataset_formatted.column_names}")

GRPO Training Configuration
Output directory: saves/llama3-3b/grpo_unsloth_3modes
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 5e-06
Generations per prompt: 4
Temperature: 0.7
Max new tokens: 4096


Map:   0%|          | 0/27137 [00:00<?, ? examples/s]

Map: 100%|██████████| 1574/1574 [00:00<00:00, 16016.21 examples/s]


✓ Formatted 27137 training examples
✓ Formatted 1574 validation examples

Dataset columns: ['prompt', 'mode', 'chosen']


In [ ]:
print("\nInitializing GRPO Trainer...")

reward_fn.__name__ = "custom_multimode_reward"
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    tokenizer=tokenizer,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    reward_funcs=[reward_fn],
    max_length=MODEL_MAX_LENGTH,
)


print("✓ GRPO Trainer initialized")
print("\n" + "="*70)
print("Starting GRPO Training with 3 Reasoning Modes!")
print("="*70)
print(f"\nTraining {len(train_dataset_formatted):,} examples")
print(f"  • Low mode:    {len(low_data):,}")
print(f"  • Medium mode: {len(medium_data):,}")
print(f"  • High mode:   {len(high_data):,}")
print(f"\nGenerating {NUM_GENERATIONS} responses per prompt")
print(f"Learning which responses match target reasoning depth\n")
print("="*70)

# Start training
trainer.train()

print("\n" + "="*70)
print("GRPO Training Completed!")
print("="*70)
print(f"Model saved to: {OUTPUT_DIR}")

The model is already on multiple devices. Skipping the move to device specified in `args`.



Initializing GRPO Trainer...
✓ GRPO Trainer initialized

Starting GRPO Training with 3 Reasoning Modes!

Training 27,137 examples
  • Low mode:    8,897
  • Medium mode: 9,450
  • High mode:   8,790

Generating 4 responses per prompt
Learning which responses match target reasoning depth



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,137 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / custom_multimode_reward / mean,rewards / custom_multimode_reward / std
1,0.000000,0.090926,0.000000,3634.375000,161.000000,4096.000000,0.875000,403.000000,161.000000,645.000000,0,0,0,0,0,0.000000,0.090926,0.000000
2,0.000000,0.090926,0.000000,3904.750000,1036.000000,4096.000000,0.937500,1036.000000,1036.000000,1036.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.090926,0.000000
3,0.000000,0.090926,0.000000,3892.000000,832.000000,4096.000000,0.937500,832.000000,832.000000,832.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.090926,0.000000


In [ ]:
model.save_lora("grpo_saved_lora")

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [ ]:
messages = [
    {"role": "system", "content": "Fast response required. Minimize reasoning time. Direct answers only."},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

## Step 7: Save the Trained Model

In [ ]:
print("Saving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")